In [ ]:
import math

import torch

SQRT_2 = math.sqrt(2)


def f(x, mu, sigma):
    r""".. math::
    Ψ_{μ,σ}(x) = (F₁⁻¹∘F₂)(x)
            &= \sqrt{2}σ⋅\erf⁻¹\Bigl(
                      ½\erf\Bigl(\frac{x+μ}{σ\sqrt{2}}\Bigr)
                    + ½\erf\Bigl(\frac{x-μ}{σ\sqrt{2}}\Bigr)
            \Bigr)
    """
    s = sigma * SQRT_2
    a = (x + mu) / s
    b = (x - mu) / s
    return s * torch.erfinv(0.5 * torch.erf(a) + 0.5 * torch.erf(b))

In [ ]:
import matplotlib.pyplot as plt

# sample inputs
x = torch.linspace(-5.0, 5.0, steps=1000)
y = f(x, mu=1.0, sigma=0.5)

plt.figure(figsize=(6, 4))
plt.plot(x.numpy(), y.numpy(), label=r"$\Psi_{1,0.5}(x)$")
plt.plot(x.numpy(), x.numpy(), "--", color="gray", label="identity")
plt.axvline(0, color="k", linewidth=0.5)
plt.axhline(0, color="k", linewidth=0.5)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Activation-like transform")
plt.legend()
plt.grid(True)
plt.tight_layout()

In [ ]:
def f_prime(x, mu, sigma):
    r""".. math::
    ½e^{(Ψ(x)/σ\sqrt{2})²}(e^{(x+μ)²/(2σ²)} + e^{(x-μ)²/(2σ²)})
    """
    s = sigma * SQRT_2
    a = (x + mu) / s
    b = (x - mu) / s
    y = f(x, mu, sigma)
    return 0.5 * torch.exp((y / s) ** 2) * (torch.exp(-(a**2)) + torch.exp(-(b**2)))

In [ ]:
# gradient check for f using torch.autograd.gradcheck
import torch
from torch.autograd import gradcheck

# use double precision for gradcheck
mu = torch.tensor(1.0, dtype=torch.double)
sigma = torch.tensor(0.5, dtype=torch.double)


def f_wrapper(x):
    # expect x to be a tensor with requires_grad=True and dtype=torch.double
    return f(x, mu=mu, sigma=sigma)


# small batch of inputs; avoid extreme values to keep erf/erfinv numerically stable
x = torch.linspace(-2.0, 2.0, steps=6, dtype=torch.double, requires_grad=True)

In [ ]:
gradcheck(f_wrapper, (x,), eps=1e-6, atol=1e-4)

In [ ]:
from torch.autograd import Function

MU = 1.0
SIGMA = 0.5


class F(Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return f(x, mu=MU, sigma=SIGMA)

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        grad_input = grad_output * f_prime(x, mu=MU, sigma=SIGMA)
        return grad_input


gradcheck(F.apply, (x,), eps=1e-6, atol=1e-4)

# Save and reuse tensors in backward

In [ ]:
from torch.autograd import Function

MU = 1.0
SIGMA = 0.5


class F(Function):
    @staticmethod
    def forward(ctx, x):
        s = SIGMA * SQRT_2
        a = (x + MU) / s
        b = (x - MU) / s
        y = torch.erfinv(0.5 * torch.erf(a) + 0.5 * torch.erf(b))
        ctx.save_for_backward(a, b, y)
        return s * y

    @staticmethod
    def backward(ctx, grad_output):
        a, b, y = ctx.saved_tensors
        y_prime = 0.5 * torch.exp(y**2) * (torch.exp(-(a**2)) + torch.exp(-(b**2)))
        return grad_output * y_prime


gradcheck(F.apply, (x,), eps=1e-6, atol=1e-4)

# gradient w.r.t. parameters

In [6]:
import math

import torch
from torch.autograd import Function, gradcheck

SQRT_2 = math.sqrt(2)


class F(Function):
    @staticmethod
    def forward(ctx, x, mu, sigma):
        s = sigma * SQRT_2
        a = (x + mu) / s
        b = (x - mu) / s
        y = torch.erfinv(0.5 * torch.erf(a) + 0.5 * torch.erf(b))
        ctx.save_for_backward(a, b, y)
        return s * y

    @staticmethod
    def backward(ctx, outer_grad):
        a, b, y = ctx.saved_tensors
        dX = 0.5 * torch.exp(y**2) * (torch.exp(-(a**2)) + torch.exp(-(b**2)))
        dMu = 0.5 * torch.exp(y**2) * (torch.exp(-(a**2)) - torch.exp(-(b**2)))
        dSigma = SQRT_2 * (
            y
            - 0.5 * torch.exp(y**2) * (a * torch.exp(-(a**2)) + b * torch.exp(-(b**2)))
        )
        return (outer_grad * dX), (outer_grad * dMu), (outer_grad * dSigma)


mu = torch.tensor(1.0, dtype=torch.double, requires_grad=True)
sigma = torch.tensor(0.5, dtype=torch.double, requires_grad=True)
x = torch.linspace(-2.0, 2.0, steps=6, dtype=torch.double, requires_grad=True)
gradcheck(F.apply, (x, mu, sigma), eps=1e-6, atol=1e-4)

True